# Summarize and plot felsenstein codes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import tskit
from IPython.display import display, SVG
from functools import reduce

In [ ]:
def clade_phi(a, b, c, n):
    """Clade correlation: a = carriers below branch, b = below branch, c = carriers."""
    return (a * n - b * c) / np.sqrt(b * (n - b) * c * (n - c))


def infer_errors(phi_obs, c, n, max_off=3, tol=1e-6):
    """Find (missed, extra) for the best branch, trying the smallest total error first."""
    if phi_obs >= 1 - tol:
        return 0, 0
    if c <= 0 or c >= n:                                   # monomorphic: phi undefined
        return np.nan, np.nan
    for total in range(1, max_off + 1):
        for missed in range(0, min(total, c - 1) + 1):    # at least 1 carrier must be below
            extra = total - missed
            a = c - missed
            b = a + extra
            if b >= n:                                     # branch can't hold n or more haplotypes
                continue
            if abs(clade_phi(a, b, c, n) - phi_obs) < tol:
                return missed, extra
    return np.nan, np.nan

def genotypes_at(ts, pos):
    """
    Return who carries the SNP at position `pos`.

    The result is a True/False array with one entry per haplotype,
    in the same order as ts.samples():
        True  = this haplotype carries the new (derived) allele
        False = it has the old (ancestral) allele
    """
    # ts.variants() steps through the SNPs in a tree sequence.
    # left/right limit it to the stretch [pos, pos + 1), i.e. just this one position.
    var = next(ts.variants(left=pos, right=pos + 1))

    # var.genotypes is one number per haplotype: 0 = ancestral, 1 = derived.
    # Comparing with 1 turns it into True/False.
    return var.genotypes == 1

def best_clade_corr(ts, pos, carriers):
    """Best clade correlation of a SNP on the tree at `pos`.
    carriers: True/False per haplotype, from genotypes_at()."""
    sample_ids = ts.samples()
    carrier_ids = sample_ids[carriers]

    # tracked_samples tells tskit to count, for every node, how many carriers are below it
    tree = ts.at(pos, tracked_samples=carrier_ids)

    n, c = len(sample_ids), len(carrier_ids)
    best = -np.inf
    for u in tree.nodes():
        if u == tree.root:
            continue                         # no branch above the root
        b = tree.num_samples(u)              # haplotypes below this branch
        a = tree.num_tracked_samples(u)      # carriers below this branch
        if b == 0 or b == n:
            continue                         # phi undefined
        best = max(best, clade_phi(a, b, c, n))
    return best

def hangs_off_root(ts, pos, carriers):
    """True if the carriers' clade in this tree hangs directly off the root."""
    tree = ts.at(pos)
    ids = ts.samples()[carriers]
    clade = reduce(tree.mrca, ids)        # the node above all the carriers
    return tree.parent(clade) == tree.root

In [ ]:
filename = "chr12_HapMapII_GRCh37_CEU500_seed1"
percent = "10"
num_bins = "25"

df = pd.read_csv(f"../data/felsenstein_{filename}_{percent}.00percent_{num_bins}bins_seed42.csv")
# reference ts from stdpopsim
ts_std = tskit.load(f"../../stdpopsim_simulations/simulations/{filename}.trees")
# masked ts from Relate
ts_masked = tskit.load(f"../../Relate_sims/masked_{filename}_{percent}.00percent_{num_bins}bins_seed42.trees")



In [ ]:
df.columns

In [ ]:
multi = df[df.true_mutations > 1]
print(f"{len(multi)} of {len(df)} usable hidden SNPs have more than one true mutation \n")
print("Number of mutations table")
print(df.true_mutations.value_counts().sort_index(), "\n")
multi = df[df.true_mutations > 1].sort_values("ALT_count")
print("Information on SNP's with more than one mutation")
print(multi[["position", "ALT_count", "ALT_count_bin", "true_mutations",
             "std_mutations", "relate_mutations"]].to_string(index=False))

In [ ]:
print("More than one mutation by bin count")
print(multi.ALT_count_bin.value_counts().sort_index())

In [ ]:
# ---------------------------------------------------------------
# 1. headline: true tree vs Relate, side by side
#    joint = the single best history; marginal = all histories added up
# ---------------------------------------------------------------
summary = pd.DataFrame({
    name: {
        "SNPs": len(df),
        # best (joint) history
        "": np.nan,                                                      # blank line
        "BEST (single best history)": np.nan,                            # section heading
        "  total log probability": df[f"{name}_log_joint"].sum(),
        "  mean log probability per SNP": df[f"{name}_log_joint"].mean(),
        "  median log probability per SNP": df[f"{name}_log_joint"].median(),
        "  mean mutations per SNP": df[f"{name}_mutations"].mean(),
        "  median mutations per SNP": df[f"{name}_mutations"].median(),
        "  % SNPs explained by 1 mutation": 100 * (df[f"{name}_mutations"] == 1).mean(),
        "  % SNPs with a back mutation": 100 * (df[f"{name}_back_mutations"] > 0).mean(),
        "  % SNPs with >1 new mutation": 100 * (df[f"{name}_new_mutations"] > 1).mean(),
        "  % true time inside chosen branch": 100 * df[f"{name}_time_inside"].mean(),
        "  median share of the marginal": df[f"{name}_best_share"].median(),
        "  % SNPs where best history < 90% of marginal": 100 * (df[f"{name}_best_share"] < 0.9).mean(),
        # # all histories (marginal)
        # " ": np.nan,                                                     # blank line
        # "MARGINAL (all histories)": np.nan,                              # section heading
        # "  total log likelihood": df[f"{name}_log_marginal"].sum(),
        # "  mean log likelihood per SNP": df[f"{name}_log_marginal"].mean(),
        # "  median log likelihood per SNP": df[f"{name}_log_marginal"].median(),
        # clade correlation
        "  ": np.nan,                                                    # blank line
        "CLADE CORRELATION": np.nan,                                     # section heading
        "  % best clade correlation = 1": 100 * np.isclose(df[f"{name}_best_clade_corr"], 1).mean(),
        "  median best clade correlation": df[f"{name}_best_clade_corr"].median(),
    }
    for name in ("std", "relate")
})
print(summary.round(3).to_string(na_rep=""))

In [ ]:
# ---------------------------------------------------------------
# 2. per SNP: how much worse is Relate than the truth?
# ---------------------------------------------------------------

# dlog = relate_log_likelihood − true_log_likelihood
gaps = pd.DataFrame({
    "best (joint)": df.dlog_best,
#     "marginal": df.dlog_marginal,
})
print("\nLog score, Relate minus truth (0 = equally good, negative = Relate worse) \n")
print(gaps.describe().round(2).to_string())
print("\n")
for col in gaps:
        print(f"  {col:13s} Relate as likely as or more than the stdpopsim (true) tree: "
          f"{100 * (gaps[col] >= -1e-6).mean():.1f}% of SNPs")


print("\n")
print(f"best history on the stdpopsim tree matches the true number of mutations: "
      f"{100 * (df.std_mutations == df.true_mutations).mean():.1f}% of SNPs")
print(f"best history on Relate's tree matches the true number of mutations: "
      f"{100 * (df.relate_mutations == df.true_mutations).mean():.1f}% of SNPs")

In [ ]:

# ---------------------------------------------------------------
# 3. how Relate's best history explains each SNP
# ---------------------------------------------------------------
perfect = (df.relate_mutations == 1)
extra_inside = (df.relate_new_mutations == 1) & (df.relate_back_mutations > 0)
split = (df.relate_new_mutations > 1) & (df.relate_back_mutations == 0)
both = (df.relate_new_mutations > 1) & (df.relate_back_mutations > 0)
outcome = pd.Series({
    "1 mutation (carriers form one clade)": perfect.mean(),
    "1 new + back mutation(s) (undo because non-carriers inside the clade)": extra_inside.mean(),
    ">1 new mutations (carriers split across clades)": split.mean(),
    "both": both.mean(),
}) * 100
print("\nHow Relate's best history explains the hidden SNPs (% of SNPs):")
print(outcome.round(1).to_string())


In [ ]:
one = df[df.relate_mutations == 1]
print(f"{len(one)} SNPs explained by 1 mutation on Relate's tree")
print(f"  non-carriers inside the clade = 0:  {100 * (one.relate_new_noncarriers_below == 0).mean():.1f}%")
print(f"  clade correlation = 1:              {100 * np.isclose(one.relate_new_clade_corr, 1).mean():.1f}%")
print(f"  true mutation time inside branch:   {100 * one.relate_time_inside.mean():.1f}%")

In [ ]:

# ---------------------------------------------------------------
# 4. Some SNPs where Relate's best history does worst
# ---------------------------------------------------------------
num_snps = 10
print(f"\n{num_snps} SNPs where Relate's best history does worst:")
print(df.nsmallest(num_snps, "dlog_best")[
    ["position", "ALT_count", "std_mutations", "relate_mutations", "relate_new_mutations",
     "relate_back_mutations", "relate_best_share", "relate_best_clade_corr", "dlog_best", "dlog_marginal"]
].round(3).to_string(index=False))

In [ ]:

# ---------------------------------------------------------------
# 4. by ALT count bin
# ---------------------------------------------------------------
by_freq = df.groupby("ALT_count_bin").agg(
    ALT_count_range=("ALT_count", lambda x: f"{x.min()}-{x.max()}"),
    SNPs=("position", "size"),
    pct_relate_1_mutation=("relate_mutations", lambda x: 100 * (x == 1).mean()),
    mean_relate_mutations=("relate_mutations", "mean"),
    median_dlog_best=("dlog_best", "median"),
    median_dlog_marginal=("dlog_marginal", "median"),
    median_relate_best_share=("relate_best_share", "median"),
    median_relate_best_clade_corr=("relate_best_clade_corr", "median"),
    pct_relate_time_inside=("relate_time_inside", lambda x: 100 * x.mean()),
)
print("\nby ALT count bin:")
print(by_freq.round(2).to_string())


In [ ]:
# ---------------------------------------------------------------
# mean vs median likelihood cost (best / joint), by ALT count bin
# ---------------------------------------------------------------
BLUE, ORANGE, INK_2 = "#257de0", "#f47230", "#3b3b3b"

stats = df.groupby("ALT_count_bin", observed=True).agg(
    lo=("ALT_count", "min"), hi=("ALT_count", "max"),
    mean=("dlog_best", "mean"), median=("dlog_best", "median"))
labels = [f"{a}" if a == b else f"{a}-{b}" for a, b in zip(stats.lo, stats.hi)]
xs = np.arange(len(stats))

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.plot(xs, stats["median"], color=BLUE, lw=2.2, marker="o", ms=5, label="median")
ax.plot(xs, stats["mean"], color=ORANGE, lw=2.2, marker="o", ms=5, label="mean")
ax.fill_between(xs, stats["median"], stats["mean"], color=ORANGE, alpha=0.12, lw=0)
ax.axhline(0, color="#474747", lw=0.8)

# second axis labels in plain "times less likely" on the right
ax_r = ax.twinx()
ax_r.set_ylim(ax.get_ylim())
k = np.arange(0, np.floor(-ax.get_ylim()[0] / np.log(10)) + 1, 5)
sup = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")
ax_r.set_yticks(-k * np.log(10), ["1×" if v == 0 else f"10{str(int(v)).translate(sup)}× less"
                                  for v in k])
ax_r.tick_params(length=0, labelsize=8, labelcolor=INK_2)
for s in ax_r.spines.values():
    s.set_visible(False)
ax_r.grid(False)

ax.set_xticks(xs, labels, rotation=60, ha="right")
ax.set_xlabel("ALT count")
ax.set_ylabel("Δ log-likelihood, best placement\n(Relate − true)")
ax.set_title("Typical vs average likelihood cost of Relate's tree", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

# PLOTS

In [ ]:
# clade corr v number of mutations relate places
SURFACE, INK, INK_2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
RELATE_C = "#eb6834"

max_shown = 10                                             # 10 or more mutations share one column
m = df.relate_mutations.clip(upper=max_shown)
cc = df.relate_best_clade_corr
counts = sorted(m.unique())
tick_labels = [f"{k}+" if k == max_shown else str(k) for k in counts]

fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)

# every SNP as a dot, spread sideways a little so overlapping dots stay visible
rng = np.random.default_rng(0)
jitter = rng.uniform(-0.25, 0.25, len(m))
ax.scatter(m + jitter, cc, s=10, color=RELATE_C, alpha=0.25, linewidth=0, zorder=2)

# the median for each number of mutations, plus how many SNPs it is based on
for k in counts:
    vals = cc[m == k]
    ax.plot([k - 0.32, k + 0.32], [vals.median()] * 2, color=INK, linewidth=2, zorder=3)
    ax.text(k, -0.08, f"n={len(vals):,}", ha="center", va="top", fontsize=8, color=MUTED)

ax.set_xticks(counts, tick_labels)
ax.set_xlabel("mutations in Relate's best history", color=INK_2, labelpad=18)
ax.set_ylabel("best clade correlation on Relate's tree", color=INK_2)
ax.set_ylim(-0.15, 1.05)
ax.set_title("Clade correlation vs number of mutations (Relate tree)",
             loc="left", fontsize=11, fontweight="bold", color=INK)
ax.text(1.0, 1.02, "dots = SNPs, black line = median", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=9, color=MUTED)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=MUTED, length=0)
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# setup shared by the plots below
# ---------------------------------------------------------------
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c2c2c2", "axes.grid": True, "grid.color": "#e7e7e7",
    "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 9,
})
BLUE, ORANGE, AQUA, GREY = "#257de0", "#f47230", "#00b894", "#8a8a8a"
SURFACE = "white"
MUT_RAMP = ["#84b9f3", "#338ced", "#1b64be", "#0e3d7e"]       # 1, 2, 3, 4+ mutations
MUT_TEXT = ["#0b0b0b", "white", "white", "white"]
OUTCOME_C = [BLUE, ORANGE, AQUA, "#f2a600"]
OUTCOME_TEXT = ["white", "white", "white", "#0b0b0b"]

bin_col = df["ALT_count_bin"]
seen = set(bin_col.dropna())
if isinstance(bin_col.dtype, pd.CategoricalDtype):
    present = [b for b in bin_col.cat.categories if b in seen]
else:
    present = sorted(seen)
groups = [df.loc[bin_col == b] for b in present]
x = np.arange(len(present))
bin_labels = []
for g in groups:
    lo, hi = g["ALT_count"].min(), g["ALT_count"].max()
    bin_labels.append(f"{lo}" if lo == hi else f"{lo}-{hi}")


def stacked(ax, share, colours, inks):
    bottom = np.zeros(len(present))
    for col, colour, ink in zip(share.columns, colours, inks):
        ax.bar(x, share[col], bottom=bottom, color=colour, width=0.7,
               edgecolor=SURFACE, linewidth=2, label=col)
        for xi, (b, h) in enumerate(zip(bottom, share[col])):
            if h >= 10:                                         # label segments big enough to read
                ax.text(xi, b + h / 2, f"{h:.0f}", ha="center", va="center", fontsize=7, color=ink)
        bottom += share[col].to_numpy()
    ax.set_ylim(0, 100)
    ax.tick_params(length=0)
    ax.grid(axis="x", visible=False)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)


# ---------------------------------------------------------------
# 3. how Relate explains the hidden SNPs, by ALT count bin
# ---------------------------------------------------------------
mut = df["relate_mutations"].clip(upper=4).map({1: "1", 2: "2", 3: "3", 4: "4+"})
mut_share = (pd.crosstab(bin_col, mut, normalize="index") * 100) \
    .reindex(index=present, columns=["1", "2", "3", "4+"]).fillna(0)

kind = np.select(
    [df.relate_mutations == 1,
     (df.relate_new_mutations == 1) & (df.relate_back_mutations > 0),
     (df.relate_new_mutations > 1) & (df.relate_back_mutations == 0)],
    ["one mutation", "non-carriers inside the clade (back mutations)",
     "carriers split across clades (extra new mutations)"],
    default="both")
order = ["one mutation", "non-carriers inside the clade (back mutations)",
         "carriers split across clades (extra new mutations)", "both"]
kind_share = (pd.crosstab(bin_col, kind, normalize="index") * 100) \
    .reindex(index=present, columns=order).fillna(0)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
stacked(ax1, mut_share, MUT_RAMP, MUT_TEXT)
ax1.set_ylabel("% of hidden SNPs")
ax1.set_title("How many mutations Relate places per hidden SNP")
ax1.get_legend().set_title("mutations")

stacked(ax2, kind_share, OUTCOME_C, OUTCOME_TEXT)
ax2.set_ylabel("% of hidden SNPs")
ax2.set_title("How Relate's tree explains each hidden SNP")
ax2.set_xticks(x, bin_labels, rotation=60, ha="right")
ax2.set_xlabel("ALT count")
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# share of SNPs within a given factor of the true tree's likelihood
# ---------------------------------------------------------------
ln10 = np.log(10)
thresholds = [("as likely or more", 0), ("within 10×", 1), ("within 100×", 2),
              ("within 10⁴×", 4), ("within 10⁶×", 6), ("within 10¹⁰×", 10), ("within 10²⁰×", 20)]

rows = []
for label, k in thresholds:
    cut = -k * ln10 - 1e-9                 # dlog ≥ −k·ln(10)  ⇔  at most 10^k times less likely
    rows.append({
        "Relate's tree is …": label,
        "best (joint) %": 100 * (df.dlog_best >= cut).mean(),
        # "marginal %": 100 * (df.dlog_marginal >= cut).mean(),
    })
within = pd.DataFrame(rows).set_index("Relate's tree is …")
print(within.round(1).to_string())

In [ ]:
# ---------------------------------------------------------------
# how much less likely is each hidden SNP on Relate's tree? (best / joint)
# ---------------------------------------------------------------
BLUE, GREY, INK_2, MUTED = "#257de0", "#bdbdbd", "#3b3b3b", "#8a8a8a"

orders = -df.dlog_best / np.log(10)          # log10( L_true / L_Relate ), best placement on each tree
edges = [-np.inf, 1e-9, 1, 2, 4, 6, 10, 20, np.inf]
names = ["as likely\nor more", "1–10×", "10–100×", "10²–10⁴×", "10⁴–10⁶×",
         "10⁶–10¹⁰×", "10¹⁰–10²⁰×", ">10²⁰×"]
colours = [GREY] + [BLUE] * (len(names) - 1)
band = pd.cut(orders, edges, labels=names, right=True)

# --- figure 1: all hidden SNPs ---
pct = band.value_counts(normalize=True).reindex(names) * 100
cum = pct.cumsum()

fig, ax = plt.subplots(figsize=(10, 4.6))
xb = np.arange(len(names))
ax.bar(xb, pct, color=colours, width=0.75, edgecolor="white", linewidth=2)
for xi, (p, c) in enumerate(zip(pct, cum)):
    ax.text(xi, p + 0.6, f"{p:.0f}%", ha="center", va="bottom", fontsize=10,
            fontweight="bold", color=INK_2)
    ax.text(xi, p + 0.6 + pct.max() * 0.07, f"{c:.0f}% so far", ha="center", va="bottom",
            fontsize=7.5, color=MUTED)
ax.set_xticks(xb, names)
ax.set_ylim(0, pct.max() * 1.3)
ax.set_xlabel("how many times less likely the SNP is on Relate's tree than on the true tree")
ax.set_ylabel("% of hidden SNPs")
ax.set_title("How much less likely Relate's best placement is each hidden SNP",
             loc="left", fontweight="bold")
ax.tick_params(axis="x", length=0)
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# mean vs median likelihood cost (best / joint), by ALT count bin
# ---------------------------------------------------------------
BLUE, ORANGE, INK_2 = "#257de0", "#f47230", "#3b3b3b"

stats = df.groupby("ALT_count_bin", observed=True).agg(
    lo=("ALT_count", "min"), hi=("ALT_count", "max"),
    mean=("dlog_best", "mean"), median=("dlog_best", "median"))
labels = [f"{a}" if a == b else f"{a}-{b}" for a, b in zip(stats.lo, stats.hi)]
xs = np.arange(len(stats))

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.plot(xs, stats["median"], color=BLUE, lw=2.2, marker="o", ms=5, label="median")
ax.plot(xs, stats["mean"], color=ORANGE, lw=2.2, marker="o", ms=5, label="mean")
ax.fill_between(xs, stats["median"], stats["mean"], color=ORANGE, alpha=0.12, lw=0)
ax.axhline(0, color="#474747", lw=0.8)

# second axis labels in plain "times less likely" on the right
ax_r = ax.twinx()
ax_r.set_ylim(ax.get_ylim())
k = np.arange(0, np.floor(-ax.get_ylim()[0] / np.log(10)) + 1, 5)
sup = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")
ax_r.set_yticks(-k * np.log(10), ["1×" if v == 0 else f"10{str(int(v)).translate(sup)}× less"
                                  for v in k])
ax_r.tick_params(length=0, labelsize=8, labelcolor=INK_2)
for s in ax_r.spines.values():
    s.set_visible(False)
ax_r.grid(False)

ax.set_xticks(xs, labels, rotation=60, ha="right")
ax.set_xlabel("ALT count")
ax.set_ylabel("Δ log-likelihood, best placement\n(Relate − true)")
ax.set_title("Typical vs average likelihood cost of Relate's tree", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="lower left")
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 4. likelihood cost of Relate's placement, by ALT count bin
# ---------------------------------------------------------------
BLUE, GREY = "#257de0", "#8a8a8a"

bin_col = df["ALT_count_bin"]
seen = set(bin_col.dropna())
present = [b for b in bin_col.cat.categories if b in seen] \
    if isinstance(bin_col.dtype, pd.CategoricalDtype) else sorted(seen)
groups = [df.loc[bin_col == b] for b in present]
x = np.arange(len(present))
bin_labels = []
for g in groups:
    lo, hi = g["ALT_count"].min(), g["ALT_count"].max()
    bin_labels.append(f"{lo}" if lo == hi else f"{lo}-{hi}")

fig, ax = plt.subplots(figsize=(12, 4.8))
data = [g["dlog_best"].dropna().to_numpy() for g in groups]
bp = ax.boxplot(data, positions=x, widths=0.55, showfliers=False, patch_artist=True,
                medianprops=dict(color="#0b0b0b", lw=1.5),
                whiskerprops=dict(color=GREY), capprops=dict(color=GREY),
                boxprops=dict(edgecolor=BLUE, lw=1))
for b in bp["boxes"]:
    b.set_facecolor(BLUE + "55")
ax.axhline(0, color="#474747", lw=0.8)
ax.set_xticks(x, bin_labels, rotation=60, ha="right")
ax.set_xlabel("ALT count")
ax.set_ylabel("Δ log-likelihood, best placement\n(Relate − true)")
ax.set_title("Likelihood cost of Relate's placement", loc="left", fontweight="bold")
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()



In [ ]:
# ---------------------------------------------------------------
# 5. likelihood cost vs number of mutations Relate places
# ---------------------------------------------------------------
BLUE, GREY = "#257de0", "#8a8a8a"
CAP = 10                                                   # lump everything above this into "10+"

m = (df["relate_mutations"] - df["std_mutations"]).clip(upper=CAP)
ks = sorted(m.dropna().unique())
data = [df.loc[m == k, "dlog_best"].dropna().to_numpy() for k in ks]
xk = np.arange(len(ks))

fig, ax = plt.subplots(figsize=(10, 4.8))
bp = ax.boxplot(data, positions=xk, widths=0.6, showfliers=False, patch_artist=True,
                medianprops=dict(color="#0b0b0b", lw=1.5),
                whiskerprops=dict(color=GREY), capprops=dict(color=GREY),
                boxprops=dict(edgecolor=BLUE, lw=1))
for b in bp["boxes"]:
    b.set_facecolor(BLUE + "55")

# how many SNPs behind each box
ymin = ax.get_ylim()[0]
for xi, d in zip(xk, data):
    ax.text(xi, ymin, f"n={len(d):,}", ha="center", va="bottom", fontsize=7.5, color=GREY)

ax.axhline(0, color="#474747", lw=0.8)
ax.set_xticks(xk, [f"{int(k)}+" if k == CAP else str(int(k)) for k in ks])
ax.set_xlabel("mutations Relate places for the hidden SNP")
ax.set_ylabel("Δ log-likelihood, best placement\n(Relate − true)")
ax.set_title("Likelihood cost of Relate's placement vs number of extra mutations it places",
             loc="left", fontweight="bold")
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

In [ ]:

# ---------------------------------------------------------------
# difference in log-likelihood = ratio of likelihoods
# ---------------------------------------------------------------
# rename these to your columns of absolute log-likelihoods (natural log, best placement)
LL_TRUE, LL_RELATE = "std_log_joint", "relate_log_joint"

ll_true = df[LL_TRUE]
ll_relate = df[LL_RELATE]
lo = np.floor(np.nanpercentile(np.r_[ll_true, ll_relate], 2))     # zoom to where 98% of SNPs sit
hi = np.ceil(max(ll_true.max(), ll_relate.max()))

fig, ax = plt.subplots(figsize=(7.5, 7))
hb = ax.hexbin(ll_true, ll_relate, gridsize=60, bins="log", cmap="Blues", mincnt=1,
               extent=(lo, hi, lo, hi), linewidths=0)

# lines of constant ratio: same vertical gap everywhere = same "times less likely"
guides = [(0, "equally likely"), (6, "10⁶× less likely"), (10, "10¹⁰× less likely")]
for k, name in guides:
    off = k * np.log(10)
    ax.plot([lo, hi], [lo - off, hi - off], color="#0b0b0b" if k == 0 else "#8a8a8a",
            lw=1.4 if k == 0 else 0.9, ls="-" if k == 0 else (0, (4, 3)))
    ax.annotate(name, (hi, hi - off), xytext=(4, 0), textcoords="offset points",
                fontsize=8.5, color="#3b3b3b", ha="left", va="center", annotation_clip=False)

# one worked example: pick a typical SNP and draw its gap
mid = ll_true.between(lo + 0.55 * (hi - lo), lo + 0.75 * (hi - lo))
ex = df.loc[(df.loc[mid, LL_RELATE] - df.loc[mid, LL_TRUE]).sub(-6 * np.log(10)).abs().idxmin()]
x0, y0 = ex[LL_TRUE], ex[LL_RELATE]
ax.annotate("", xy=(x0, y0), xytext=(x0, x0),
            arrowprops=dict(arrowstyle="<->", color="#f47230", lw=1.8))
ax.plot([x0, x0], [x0, y0], "o", color="#f47230", ms=4)
ax.annotate(f"gap = {y0 - x0:.1f}\n= ln( L_Relate / L_true )\n≈ {abs(y0 - x0) / np.log(10):.0f} "
            "orders of magnitude",
            ((x0), (x0 + y0) / 2), xytext=(-12, 0), textcoords="offset points",
            fontsize=8.5, color="#3b3b3b", va="center", ha="right",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.9))

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")
ax.set_xlabel("log-likelihood on the true tree")
ax.set_ylabel("log-likelihood on Relate's tree")
ax.set_title("Vertical distance from the diagonal is the likelihood ratio",
             loc="left", fontweight="bold")
ax.grid(False)
fig.colorbar(hb, ax=ax, shrink=0.6, pad=0.22, label="hidden SNPs (log)")
fig.tight_layout()
plt.show()

# LD Score

In [ ]:
# load results filename previously calculated
results_filename = "chr12_HapMapII_GRCh37_CEU500_seed1_10.00percent500000bp_25bins_seed42"

results = pd.read_csv(f"../../masked_recovery_comp/{results_filename}_results.csv")
results

In [ ]:
# recall that these are both pearson corrs to the best snp
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(results["max_corr"], results["optimized_max_r2"], s=4, alpha=0.2)
ax.set_xlabel("max_corr")
ax.set_ylabel("max_r2")
ax.set_title("maximum clade correlation v max single r^2 value")
x = np.linspace(0, 1, 200)
box = dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="grey", alpha=0.9)
# ax.text(0.03, 0.80,
#             "Above the curve: a neighbouring SNP\n"
#             "matches the data better than any clade.\n"
#             "The pattern is in the data, but not in the tree.",
#             fontsize=9, va="top", bbox=box)

# ax.text(0.62, 0.10,
#         "Below the curve: the tree has a clade\n"
#         "that matches better than any neighbor SNP.",
#         fontsize=9, va="center", bbox=box)
# # the tree is combining info from sveral snps to reconstruct


ax.plot(x, x**2, color="red", linewidth=1, label=r"max_r2 = max_corr$^2$")
ax.legend()

ax.annotate(r"On the curve: max_r2 = max_corr$^2$" "\n"
            "most informative neighbor SNP is exactly the best clade",
            xy=(0.75, 0.5625), xytext=(0.30, 0.45),
            fontsize=9, color="red", bbox=box,
            arrowprops=dict(arrowstyle="->", color="red"))


fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

heatmap = ax.hexbin(
    results["max_corr"],
    results["optimized_top10_r2_sum"],
    gridsize=40,
    bins="log",
    mincnt=1,
    cmap="magma_r"
)
# n = 1000   # number of haplotypes in this analysis
# for phi, label in [(np.sqrt((n - 2) / (2 * (n - 1))), "1 of 2"),
#                    (np.sqrt(2 * (n - 3) / (3 * (n - 2))), "2 of 3"),
#                    (np.sqrt((n - 3) / (3 * (n - 1))), "1 of 3")]:
#     ax.axvline(phi, color="black", linewidth=0.8, linestyle="--")
#     ax.text(phi, ax.get_ylim()[1], label, ha="center", va="bottom", fontsize=8)

fig.colorbar(heatmap, ax=ax, label="Number of SNPs")

ax.set_xlabel("Best clade correlation")
ax.set_ylabel(r"Sum of top 10 $r^2$ values")
ax.set_title(r"Top 10 $r^2$ value for SNP by clade correlation in 1MB window")

fig.tight_layout()
fig.savefig("top10_r2_clade_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

heatmap = ax.hexbin(
    results["max_corr"],
    results["optimized_top10_r2_sum"],
    gridsize=40,
    bins="log",
    mincnt=1,
    cmap="magma_r"
)

n = 1000   # number of haplotypes in this analysis


def phi_nested(small, big, n):
    """Clade correlation when one set sits inside the other (sizes small <= big)."""
    return np.sqrt(small * (n - big) / (big * (n - small)))


stripes = [
    (phi_nested(1, 3, n), "tripleton,\n2 missed"),
    (phi_nested(1, 2, n), "doubleton,\n1 missed"),
    (phi_nested(2, 3, n), "tripleton, 1 missed\ndoubleton, 1 extra"),
]

for phi, label in stripes:
    ax.axvline(phi, color="black", linewidth=0.8, linestyle="--")
    ax.text(phi, ax.get_ylim()[1], label, ha="center", va="bottom",
            fontsize=8, linespacing=1.1)

fig.colorbar(heatmap, ax=ax, label="Number of SNPs")

ax.set_xlabel("Best clade correlation")
ax.set_ylabel(r"Sum of top 10 $r^2$ values")
ax.set_title(r"Top 10 $r^2$ value for SNP by clade correlation in 1MB window", pad=28)

fig.tight_layout()
fig.savefig("top10_r2_clade_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
data = results                      # one row per masked SNP
x_col = "max_corr"          # best clade correlation
y_col = "optimized_top10_r2_sum"             # sum of the top 10 r^2 values
count_col = "ALT_count"            # number of carriers
n = 1000                            # number of haplotypes

groups = [
    ("rare: 1-4 carriers", data[data[count_col] <= 4]),
    ("5 or more carriers", data[data[count_col] >= 5]),
]

# predicted stripe positions for rare variants (best clade = some of the carriers, or 1 carrier + 1 non-carrier)
predicted = {
    "1 of 2": np.sqrt((n - 2) / (2 * (n - 1))),
    "2 of 3": np.sqrt(2 * (n - 3) / (3 * (n - 2))),
    "1 of 3": np.sqrt((n - 3) / (3 * (n - 1))),
    "3 of 4": np.sqrt(3 * (n - 4) / (4 * (n - 3))),
    "1 of 4": np.sqrt((n - 4) / (4 * (n - 1))),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True, layout="constrained")
norm = LogNorm(vmin=1, vmax=None)                  # same colour scale on both panels
hbs = []
for ax, (title, d) in zip(axes, groups):
    hb = ax.hexbin(d[x_col], d[y_col], gridsize=40, extent=(0, 1, 0, 10),
                   cmap="magma_r", norm=norm, mincnt=1)
    hbs.append(hb)
    if ax is axes[0]:                              # predicted lines on the rare panel only
        for i, (label, phi) in enumerate(sorted(predicted.items(), key=lambda kv: kv[1])):
            ax.axvline(phi, color="black", linewidth=0.7, linestyle="--", alpha=0.6)
            ax.text(phi, 10.3 + 0.45 * (i % 2), label, ha="center", va="bottom", fontsize=7)   # staggered so close labels don't overlap
    ax.set_title(f"{title}  (n = {len(d):,} SNPs)")
    ax.set_ylim(-0.5, 11.3)
    ax.set_xlabel("Best clade correlation")
axes[0].set_ylabel("Sum of top 10 $r^2$ values")

# one colour bar, scaled to the busier panel
vmax = max(h.get_array().max() for h in hbs)
for h in hbs:
    h.set_norm(LogNorm(vmin=1, vmax=vmax))
fig.colorbar(hbs[0], ax=axes, label="Number of SNPs")
fig.suptitle("Top 10 $r^2$ by best clade correlation, split by number of carriers")
plt.show()

# Off by 1 stuff

In [ ]:
results[["missed", "extra"]] = results.apply(
    lambda r: pd.Series(infer_errors(r["max_corr"], int(r["ALT_count"]), n)), axis=1)
results["off_by"] = results["missed"] + results["extra"]   # NaN = more than 3 off

In [ ]:
off = results["off_by"].fillna(4).clip(upper=4)
counts = off.value_counts().sort_index()
off_by_summary = pd.DataFrame({
    "SNPs": counts,
    "percent": (100 * counts / counts.sum()).round(1),
}).rename(index={0: "0", 1: "1", 2: "2", 3: "3", 4: "4+"})
off_by_summary.index.name = "off_by"
print(off_by_summary)


In [ ]:
# split into missed vs extra
pd.crosstab(results["missed"], results["extra"], margins=True)

In [ ]:
# off-by counts for each allele count (5+ grouped together)
carriers = results["ALT_count"].clip(upper=5).replace(5, "5+")
pd.crosstab(carriers, results["off_by"].fillna(4).replace(4, "4+"), margins=True)

# Whats with the hight r^2 low clade corr ?

In [ ]:
odd = results[(results["optimized_top10_r2_sum"] > 9.99) & (results["max_corr"] < 0.9)]
print(len(odd), "SNPs with 10 perfect LD partners but clade correlation < 0.9")

print("ALT_count distribution")
print(odd["ALT_count"].describe())

In [ ]:
odd[["SNP position", "ALT_count", "max_corr"]].sort_values("max_corr").head(20)

In [ ]:
odd = odd.copy()
odd["group"] = pd.cut(odd["ALT_count"], [0, 10, 500, 992, 1000],
                      labels=["rare (≤10)", "11–500", "501–992", "near-fixed (993+)"])
display(odd["group"].value_counts().sort_index())

In [ ]:
# ---------- is each SNP root-adjacent in the true tree? ----------

def root_adjacent_many(ts, positions):
    """For each position, True if one of the root's children has exactly the carriers below it.
    Steps along the genome once, so it's fast for many SNPs."""
    wanted = set(int(round(p)) for p in positions)
    samples = ts.samples()
    tree = ts.first()
    out = {}
    for var in ts.variants():                          # every SNP, left to right
        p = int(round(var.site.position))
        if p not in wanted:
            continue
        tree.seek(var.site.position)                   # cheap: only moves forward
        carriers = samples[var.genotypes == 1]
        c = len(carriers)
        out[p] = False
        for child in tree.children(tree.root):
            if tree.num_samples(child) == c:           # right size: now check it's the carriers
                if set(tree.samples(child)) == set(carriers):
                    out[p] = True
                break
    return out


# ---------- label every masked SNP ----------

root_flags = root_adjacent_many(ts_std, results["SNP position"])     # all masked SNPs, a few minutes

bins = [0, 10, 500, 992, 1000]
labels = ["rare (≤10)", "11–500", "501–992", "near-fixed (993+)"]

snps = results.copy()
snps["group"] = pd.cut(snps["ALT_count"], bins, labels=labels)
snps["root_adjacent"] = snps["SNP position"].round().astype(int).map(root_flags)

missing = snps["root_adjacent"].isna().sum()
print(f"SNPs checked: {len(snps):,}   with no answer: {missing}")      # should be 0
assert missing == 0, "some SNPs weren't found in ts_std"

odd = snps[(snps["optimized_top10_r2_sum"] > 9.99) & (snps["max_corr"] < 0.9)]


# ---------- one chart for one set of SNPs, with an "all" bar ----------

def plot_root_adjacent(df, title, color, all_color):
    g = df.groupby("group", observed=False)["root_adjacent"]
    k, n = g.sum(), g.size()

    k.index = k.index.astype(str)
    n.index = n.index.astype(str)
    k["all"], n["all"] = k.sum(), n.sum()                 # the overall bar
    pct = 100 * k / n

    colors = [color] * (len(pct) - 1) + [all_color]
    fig, ax = plt.subplots(figsize=(9, 3.5))
    bars = ax.bar(pct.index, pct, color=colors)
    ax.bar_label(bars, labels=[f"{p:.0f}%\n({int(a):,} of {int(b):,})"
                               for p, a, b in zip(pct, k, n)], fontsize=9)
    ax.axvline(len(pct) - 1.5, color="grey", linewidth=0.8, linestyle=":")   # separates "all"
    ax.set_ylim(0, 115)
    ax.set_ylabel("% next to the true root")
    ax.set_xlabel("number of carriers")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_root_adjacent(odd,
                   "Odd SNPs (10 perfect LD partners, clade correlation < 0.9):\n"
                   "how many sit next to the true root?",
                   "#dd8452", "#a85a28")

plot_root_adjacent(snps,
                   "All masked SNPs: how many sit next to the true root?",
                   "#999999", "#5f5e5a")

# ---------- do root-adjacent SNPs usually have 10 perfect LD partners? ----------

adj = snps[snps["root_adjacent"]]
perfect10 = adj["optimized_top10_r2_sum"] > 9.99
print(f"\nroot-adjacent SNPs: {len(adj):,}")
print(f"  with 10 perfect LD partners: {perfect10.sum():,} ({100 * perfect10.mean():.0f}%)")

In [ ]:
pos = 83307140
tree_std = ts_std.at(pos)
tree_masked = ts_masked.at(pos)

sample_ids = ts_std.samples()
carrier = genotypes_at(ts_std, pos)              # True = carries the SNP
noncarrier_nodes = sample_ids[~carrier]
print(f"{carrier.sum()} carriers, non-carriers: {list(noncarrier_nodes)}")

wide_fmt = (2000, 600)
node_label_style = (
    ".node > .lab {font-size: 80%}"
    ".leaf > .lab {text-anchor: start; transform: rotate(90deg) translate(6px)}"
)

# label only the non-carrier(s)
node_labels = {int(u): str(u) for u in noncarrier_nodes}


def highlight_style(tree):
    style = ".leaf > .lab {font-size: 12px;}"
    for node in noncarrier_nodes:
        style += f"""
        .node.n{int(node)} > .sym {{fill: crimson; stroke: crimson; r: 5px;}}
        .node.n{int(node)} > .lab {{fill: crimson; font-weight: bold;}}
        .node.n{int(node)} > .edge {{stroke: crimson; stroke-width: 3px;}}
        """
    style += f".node.n{tree.root} > .sym {{fill: royalblue; r: 6px;}}"   # root in blue
    return style


display(SVG(tree_masked.draw_svg(
    title="Relate tree (masked) — red = haplotype without the SNP, blue = root",
    size=wide_fmt,
    node_labels=node_labels,
    time_scale="log_time",
    style=node_label_style + highlight_style(tree_masked),
)))

display(SVG(tree_std.draw_svg(
    title="True tree (stdpopsim) — red = haplotype without the SNP, blue = root",
    size=wide_fmt,
    node_labels=node_labels,
    time_scale="log_time",
    style=node_label_style + highlight_style(tree_std),
)))

#### not now

Hangs off the root" is always about the true tree.

It describes where the mutation really happened: in the true tree, is the branch carrying the mutation one of the two branches coming straight out of the root?

In [ ]:
# # ---------- classify each of the 123 SNPs ----------

# rows = []
# for pos, group in zip(odd["SNP position"], odd["group"]):
#     x = genotypes_at(ts_std, pos)                           # True = carrier
#     rows.append(dict(
#         position=pos,
#         group=group,
#         ALT_count=int(x.sum()),
#         clade_corr=best_clade_corr(ts_masked, pos, x),      # should match max_corr
#         # split present: the non-carriers form a clade in Relate's tree
#         split_present=best_clade_corr(ts_masked, pos, ~x) > 1 - 1e-9,
#         # in the true tree, the mutation's branch comes straight off the root
#         root_adjacent=hangs_off_root(ts_std, pos, x),
#     ))
# check = pd.DataFrame(rows)

# check["outcome"] = np.select(
#     [check["split_present"], check["root_adjacent"]],
#     ["rooting error", "topology error, root-adjacent"],
#     default="topology error, not root-adjacent",
# )


In [ ]:
# rooting = check[check["split_present"]]
# print(f"rooting errors: {len(rooting)}, "
#       f"of which root-adjacent: {rooting['root_adjacent'].sum()} "
#       f"({100 * rooting['root_adjacent'].mean():.0f}%)")

In [ ]:
# check["outcome"]

In [ ]:
# # ---------- one table: count (percent of row) ----------
# # 

# order = ["rare (≤10)", "11–500", "501–992", "near-fixed (993+)"]
# cols = ["rooting error (split present, clade missing)", "topology error, root-adjacent", "topology error, not root-adjacent"]

# counts = (pd.crosstab(check["group"], check["outcome"])
#           .reindex(index=order, columns=cols, fill_value=0))
# counts.loc["all"] = counts.sum()

# pct = 100 * counts.div(counts.sum(axis=1), axis=0)
# table = counts.astype(str) + " (" + pct.round(0).astype(int).astype(str) + "%)"
# table["total"] = counts.sum(axis=1)
# display(table)



In [ ]:

# ax = counts.drop(index="all").plot.barh(stacked=True, figsize=(10, 4),
#                                         color=["#4c72b0", "#dd8452", "#999999"])
# ax.invert_yaxis()
# ax.set_xlabel("SNPs")
# ax.set_ylabel("")
# ax.set_title("SNPs with 10 perfect LD partners but clade correlation < 0.9")
# ax.legend(title="", loc="upper right", frameon=False, fontsize=9)
# plt.tight_layout()
# plt.show()

In [ ]:
# rows = []
# for pos, group in zip(odd["SNP position"], odd["group"]):
#     x = genotypes_at(ts_std, pos)            # True = carrier
#     rows.append(dict(
#         position=pos,
#         group=group,
#         ALT_count=int(x.sum()),
#         phi_carriers=best_clade_corr(ts_masked, pos, x),       # should match max_corr
#         phi_noncarriers=best_clade_corr(ts_masked, pos, ~x),   # ~x flips True/False
#     ))
# check = pd.DataFrame(rows)
# check["right_split_wrong_root"] = check["phi_noncarriers"] > 1 - 1e-9

# print(pd.crosstab(check["group"], check["right_split_wrong_root"], margins=True))
# check["true_hangs_off_root"] = [
#     hangs_off_root(ts_std, pos, genotypes_at(ts_std, pos)) for pos in check["position"]
# ]
# display(pd.crosstab([check["group"], check["right_split_wrong_root"]],
#                   check["true_hangs_off_root"], margins=True))

# check["outcome"] = np.select(
#     [check["right_split_wrong_root"], check["true_hangs_off_root"]],
#     ["right grouping, wrong root (hangs off the root)",          
#      "wrong grouping, hangs off the root"],
#     default="wrong grouping, doesn't hang off the root",
# )

# order = ["rare (≤10)", "11–500", "501–992", "near-fixed (993+)"]
# cols = ["right grouping, wrong root (hangs off the root)",     
#         "wrong grouping, hangs off the root",
#         "wrong grouping, doesn't hang off the root"]

# counts = (pd.crosstab(check["group"], check["outcome"])
#           .reindex(index=order, columns=cols, fill_value=0))
# counts.loc["all"] = counts.sum()

# # show each cell as "count (percent of its row)"
# pct = 100 * counts.div(counts.sum(axis=1), axis=0)
# table = counts.astype(str) + " (" + pct.round(0).astype(int).astype(str) + "%)"
# table["total"] = counts.sum(axis=1)
# display(table)

# ax = counts.drop(index="all").plot.barh(stacked=True, figsize=(10, 4),
#                                         color=["#4c72b0", "#dd8452", "#999999"])
# ax.invert_yaxis()
# ax.set_xlabel("SNPs")
# ax.set_ylabel("")
# ax.set_title("SNPs with 10 perfect LD partners but clade correlation < 0.9")
# ax.legend(title="", loc="upper right", frameon=False, fontsize=9)
# plt.tight_layout()
# plt.show()

In [ ]:
# pos = 83307140
# tree_std = ts_std.at(pos)
# tree_masked = ts_masked.at(pos)

# sample_ids = ts_std.samples()
# carrier = genotypes_at(ts_std, pos)              # True = carries the SNP
# noncarrier_nodes = sample_ids[~carrier]
# print(f"{carrier.sum()} carriers, non-carriers: {list(noncarrier_nodes)}")

# wide_fmt = (2000, 600)
# node_label_style = (
#     ".node > .lab {font-size: 80%}"
#     ".leaf > .lab {text-anchor: start; transform: rotate(90deg) translate(6px)}"
# )

# # label only the non-carrier(s)
# node_labels = {int(u): str(u) for u in noncarrier_nodes}


# def highlight_style(tree):
#     style = ".leaf > .lab {font-size: 12px;}"
#     for node in noncarrier_nodes:
#         style += f"""
#         .node.n{int(node)} > .sym {{fill: crimson; stroke: crimson; r: 5px;}}
#         .node.n{int(node)} > .lab {{fill: crimson; font-weight: bold;}}
#         .node.n{int(node)} > .edge {{stroke: crimson; stroke-width: 3px;}}
#         """
#     style += f".node.n{tree.root} > .sym {{fill: royalblue; r: 6px;}}"   # root in blue
#     return style


# display(SVG(tree_masked.draw_svg(
#     title="Relate tree (masked) — red = haplotype without the SNP, blue = root",
#     size=wide_fmt,
#     node_labels=node_labels,
#     time_scale="log_time",
#     style=node_label_style + highlight_style(tree_masked),
# )))

# display(SVG(tree_std.draw_svg(
#     title="True tree (stdpopsim) — red = haplotype without the SNP, blue = root",
#     size=wide_fmt,
#     node_labels=node_labels,
#     time_scale="log_time",
#     style=node_label_style + highlight_style(tree_std),
# )))

In [ ]:
# COMPARE_TO_ALL = True        # False = only the 123 odd SNPs


# # ---------- is each SNP root-adjacent in the true tree? ----------

# def root_adjacent_many(ts, positions):
#     """For each position, True if one of the root's children has exactly the carriers below it.
#     Steps along the genome once, so it's fast for many SNPs."""
#     wanted = set(int(round(p)) for p in positions)
#     samples = ts.samples()
#     tree = ts.first()
#     out = {}
#     for var in ts.variants():                          # every SNP, left to right
#         p = int(round(var.site.position))
#         if p not in wanted:
#             continue
#         tree.seek(var.site.position)                   # cheap: only moves forward
#         carriers = samples[var.genotypes == 1]
#         c = len(carriers)
#         out[p] = False
#         for child in tree.children(tree.root):
#             if tree.num_samples(child) == c:           # right size: now check it's the carriers
#                 if set(tree.samples(child)) == set(carriers):
#                     out[p] = True
#                 break
#     return out


# # ---------- which SNPs ----------

# odd = results[(results["optimized_top10_r2_sum"] > 9.99) & (results["max_corr"] < 0.9)]
# snps_to_check = results if COMPARE_TO_ALL else odd

# flags = root_adjacent_many(ts_std, snps_to_check["SNP position"])    # one pass along the genome


# # ---------- count root-adjacent SNPs per frequency group ----------

# bins = [0, 10, 500, 992, 1000]
# labels = ["rare (≤10)", "11–500", "501–992", "near-fixed (993+)"]


# def root_adjacent_table(df, name):
#     d = df[["SNP position", "ALT_count"]].copy()
#     d["group"] = pd.cut(d["ALT_count"], bins, labels=labels)
#     d["root_adjacent"] = d["SNP position"].round().astype(int).map(flags)

#     t = d.groupby("group", observed=False)["root_adjacent"].agg(SNPs="size", root_adjacent="sum")
#     t.index = t.index.astype(str)
#     t.loc["all"] = [t["SNPs"].sum(), t["root_adjacent"].sum()]
#     t["percent"] = (100 * t["root_adjacent"] / t["SNPs"]).round(1)
#     print(name)
#     display(t)
#     return t


# odd_t = root_adjacent_table(odd, "10 perfect LD partners, clade correlation < 0.9")
# if COMPARE_TO_ALL:
#     all_t = root_adjacent_table(results, "all masked SNPs")


# # ---------- chart ----------

# plot = pd.DataFrame({"10 perfect partners, clade corr < 0.9": odd_t["percent"]})
# if COMPARE_TO_ALL:
#     plot["all masked SNPs"] = all_t["percent"]

# ax = plot.plot.bar(figsize=(10, 4), rot=0, color=["#dd8452", "#999999"][:plot.shape[1]])
# ax.set_ylabel("% root-adjacent in the true tree")
# ax.set_xlabel("")
# ax.set_ylim(0, 105)
# ax.set_title("How many SNPs sit on a branch coming straight off the true root?")
# for bars in ax.containers:
#     ax.bar_label(bars, fmt="%.0f%%", fontsize=8)
# ax.legend(frameon=False, fontsize=9)
# plt.tight_layout()
# plt.show()